# Notebook 01 — CNN Baseline + MobileNetV2 Student + ResNet50 Teacher

**NeuroDriver CNN ADAS Colombia** — Phase 1 milestone (updated 2026-09-22).

Designed to run on Google Colab (mount Drive / clone repo) but portable to local Windows/Linux.
Full training, Colombian fine-tuning, and Knowledge Distillation are **not** executed in this
notebook — see Sections 12-14 for what is prepared vs. pending.

**Updated 2026-09-22:** models train on two independent binary targets (`has_vehicle`,
`has_pedestrian`) via `BinaryCrossentropy(from_logits=True)`, not 4-class softmax — real data showed
severe imbalance for pure PEDESTRIAN frames. The Teacher/Student ladder now also includes a future
ResNet50 Teacher alongside the two Students (MobileNetV2, lightweight CNN). See
`docs/decisions_log.md`.


## 1. Environment / GPU check

In [ ]:
import subprocess, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "00_check_environment.py")])


## 2. Deterministic seeds

In [ ]:
from neurodriver_cnn.utils.seed import set_global_seed
from neurodriver_cnn.config import load_dataset_config, load_training_config

SEED = load_dataset_config(PROJECT_ROOT)["seed"]
set_global_seed(SEED)
print(f"Seed set to {SEED}")


## 3. Configurable project / data root

No personal/hardcoded Google Drive paths — override `DATA_ROOT` below only if your data lives outside `PROJECT_ROOT/data`.

In [ ]:
DATA_ROOT = PROJECT_ROOT / "data"
MANIFEST_PATH = DATA_ROOT / "processed" / "manifests" / "experiment_manifest.csv"
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"MANIFEST_PATH = {MANIFEST_PATH}")


## 4. Load Common Manifest + label validation

In [ ]:
import pandas as pd
from neurodriver_cnn.data.manifest import validate_manifest_invariants

if MANIFEST_PATH.exists():
    manifest_df = pd.read_csv(MANIFEST_PATH)
    violations = validate_manifest_invariants(manifest_df)
    print(f"Loaded {len(manifest_df)} rows.")
    if violations:
        print("VIOLATIONS:")
        for v in violations:
            print(f" - {v}")
    else:
        print("[OK] Manifest passed invariant validation.")
else:
    manifest_df = None
    print("PENDING: experiment_manifest.csv not found.")
    print("Run scripts/00-04 after placing the BDD100K .tar (see docs/bdd100k_setup.md).")


## 5. tf.data pipeline (TRAIN/VAL/TEST)

**Multi-label targets:** each example yields `(image, [has_vehicle, has_pedestrian])` as a
2-element float32 vector — matching the models' two-logit output — instead of a single 4-class
integer label.


In [ ]:
import tensorflow as tf

IMAGE_SIZE = tuple(load_dataset_config(PROJECT_ROOT)["image_size"])
BATCH_SIZE = load_training_config(PROJECT_ROOT)["batch_size"]


def load_image(path, targets):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    return image, targets


def make_dataset(df: pd.DataFrame, split: str, shuffle: bool) -> tf.data.Dataset:
    split_df = df[df["split"] == split]
    paths = split_df["image_path"].tolist()
    targets = split_df[["has_vehicle", "has_pedestrian"]].astype("float32").values
    ds = tf.data.Dataset.from_tensor_slices((paths, targets))
    if shuffle:
        ds = ds.shuffle(buffer_size=max(1, len(paths)), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


if manifest_df is not None:
    train_ds = make_dataset(manifest_df, "TRAIN", shuffle=True)
    val_ds = make_dataset(manifest_df, "VALIDATION", shuffle=False)
    test_ds = make_dataset(manifest_df, "TEST", shuffle=False)
    print("[OK] tf.data pipelines built (2-target multi-label).")
else:
    train_ds = val_ds = test_ds = None
    print("PENDING: cannot build tf.data pipelines without a manifest.")


## 6. Train-only augmentation

Applied **only** to `train_ds`, after the split (no leakage): horizontal flip, modest
brightness/contrast, modest zoom. **Never** vertical flip or 90/180-degree rotation — the current
labels carry no left/right semantics that a horizontal flip would violate, but a vertical flip or
road rotation would produce physically nonsensical driving scenes.


In [ ]:
augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomBrightness(0.1),
        tf.keras.layers.RandomContrast(0.1),
        tf.keras.layers.RandomZoom(0.1),
    ],
    name="train_augmentation",
)


def augment(image, targets):
    return augmentation(image, training=True), targets


if train_ds is not None:
    train_ds = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    print("[OK] Augmentation attached to train_ds only.")


## 7. Simple CNN baseline (Student 2 in the future KD ladder)

In [ ]:
from neurodriver_cnn.models.baseline import build_baseline_cnn, compile_baseline, build_baseline_callbacks
from neurodriver_cnn.evaluation.metrics import count_parameters

training_config = load_training_config(PROJECT_ROOT)

baseline_model = build_baseline_cnn(
    input_shape=IMAGE_SIZE + (3,), dropout=training_config["baseline"]["dropout"]
)
compile_baseline(baseline_model, learning_rate=training_config["baseline"]["learning_rate"])
baseline_model.summary()

baseline_params = count_parameters(baseline_model)
print(baseline_params)
assert baseline_params["total_params"] < 1_000_000, (
    f"Baseline CNN (Student 2) must stay lightweight (<1M params), got {baseline_params['total_params']}"
)
print(f"[OK] Baseline CNN is lightweight: {baseline_params['total_params']:,} total params (<1,000,000).")


## 8. Callbacks + baseline training

In [ ]:
callbacks = build_baseline_callbacks(
    patience_es=training_config["callbacks"]["early_stopping_patience"],
    patience_lr=training_config["callbacks"]["reduce_lr_patience"],
)

BASELINE_STATUS = "PENDING"
if train_ds is not None and val_ds is not None:
    # A short run is acceptable for honest preliminary metrics; NOT a long
    # production training run. Increase epochs deliberately, not automatically.
    history = baseline_model.fit(
        train_ds, validation_data=val_ds, epochs=training_config["baseline"]["epochs"], callbacks=callbacks
    )
    BASELINE_STATUS = "DONE (preliminary short run)"
else:
    print("PENDING: baseline training skipped, no data available yet.")

print(f"Baseline status: {BASELINE_STATUS}")


## 9. Baseline evaluation

Uses `labels_from_logits` to threshold the two sigmoid probabilities and derive the 4-state ADAS
label, then reports both the per-target (vehicle/pedestrian) binary metrics and the derived 4-state
classification metrics/confusion matrix — never Accuracy alone.


In [ ]:
from neurodriver_cnn.evaluation.metrics import (
    CLASS_NAME_TO_INDEX, classification_metrics, confusion_matrix, evaluate_motorcycle_subset,
    labels_from_logits, multilabel_binary_metrics,
)
from neurodriver_cnn.labeling.frame_labels import derive_label
import numpy as np

LABEL_THRESHOLD = load_training_config(PROJECT_ROOT)["evaluation"]["label_threshold"]

if test_ds is not None and BASELINE_STATUS.startswith("DONE"):
    y_true = np.concatenate([y.numpy() for _, y in test_ds])  # shape (N, 2): [has_vehicle, has_pedestrian]
    logits = baseline_model.predict(test_ds)  # shape (N, 2): [vehicle_logit, pedestrian_logit]

    pred = labels_from_logits(logits[:, 0], logits[:, 1], threshold=LABEL_THRESHOLD)
    true_labels = np.array(
        [CLASS_NAME_TO_INDEX[derive_label(bool(v), bool(p))] for v, p in y_true.astype(bool)]
    )

    print("Per-target binary metrics:")
    print(multilabel_binary_metrics(y_true[:, 0], y_true[:, 1], pred["has_vehicle"], pred["has_pedestrian"]))
    print("\nDerived 4-state metrics:")
    print(classification_metrics(true_labels, pred["label"]))
    print(confusion_matrix(true_labels, pred["label"]))
else:
    print("PENDING: baseline evaluation requires a trained baseline and TEST data.")


## 10. MobileNetV2 Student construction (Student 1)

In [ ]:
from neurodriver_cnn.models.mobilenetv2 import build_mobilenetv2_student, compile_student
from neurodriver_cnn.evaluation.metrics import count_parameters

student_model, student_base_model = build_mobilenetv2_student(
    input_shape=IMAGE_SIZE + (3,), dropout=training_config["mobilenetv2"]["dropout"], freeze_backbone=True
)
compile_student(student_model, learning_rate=training_config["mobilenetv2"]["head_learning_rate"])
student_model.summary()
print(count_parameters(student_model))


## 11. MobileNetV2 forward-pass smoke test

In [ ]:
MOBILENET_SMOKE_TEST_STATUS = "PENDING"
if train_ds is not None:
    for images, targets in train_ds.take(1):
        logits = student_model.predict(images, verbose=0)
        assert logits.shape == (images.shape[0], 2), logits.shape  # [vehicle_logit, pedestrian_logit]
        assert not np.isnan(logits).any(), "NaN in student logits"
        MOBILENET_SMOKE_TEST_STATUS = "DONE"
        print(f"[OK] logits shape={logits.shape}, no NaNs.")
else:
    print("PENDING: forward-pass smoke test requires real data (a real batch), not synthetic tensors.")

print(f"MobileNetV2 smoke-test status: {MOBILENET_SMOKE_TEST_STATUS}")


## 12. ResNet50 Teacher construction (architecture only, not trained)

Future Knowledge Distillation Teacher. Same two-logit output space as both Students, so no adapter
layer is needed to compare Teacher/Student logits later (see `docs/teacher_student_contract.md`).
Not trained in this phase — construction + parameter count + a forward-pass smoke test only.


In [ ]:
from neurodriver_cnn.models.resnet50_teacher import build_resnet50_teacher, compile_teacher

RESNET50_STATUS = "PENDING"
if train_ds is not None:
    teacher_model, teacher_base_model = build_resnet50_teacher(
        input_shape=IMAGE_SIZE + (3,),
        dropout=training_config["resnet50_teacher"]["dropout"],
        freeze_backbone=True,
    )
    compile_teacher(teacher_model, learning_rate=training_config["resnet50_teacher"]["head_learning_rate"])
    print(count_parameters(teacher_model))

    for images, targets in train_ds.take(1):
        teacher_logits = teacher_model.predict(images, verbose=0)
        assert teacher_logits.shape == (images.shape[0], 2), teacher_logits.shape
        assert not np.isnan(teacher_logits).any(), "NaN in teacher logits"
        RESNET50_STATUS = "DONE (architecture + smoke test only, not trained)"
        print(f"[OK] Teacher logits shape={teacher_logits.shape}, no NaNs.")
else:
    print("PENDING: ResNet50 Teacher smoke test requires real data.")

print(f"ResNet50 Teacher status: {RESNET50_STATUS}")


## 13. Future fine-tuning cells (prepared, not executed)

Fine-tuning procedure once BDD supervised training on the head is solid (applies to both Students):

1. train the head with the backbone frozen (Sections 7-11, done above);
2. `unfreeze_for_fine_tuning(student_base_model, unfreeze_from_layer=...)`;
3. recompile with a low learning rate (`training_config["mobilenetv2"]["fine_tune_learning_rate"]`, ~1e-5);
4. fine-tune carefully; BatchNormalization layers are kept frozen (inference mode) even when
   unfrozen, since small BDD fine-tuning batches are not representative enough to safely update
   BatchNorm running statistics;
5. keep BDD source-domain fine-tuning and later Colombian target-domain fine-tuning as **separate,
   comparable experiments** (see `docs/colombian_domain_strategy.md`).


In [ ]:
from neurodriver_cnn.models.mobilenetv2 import unfreeze_for_fine_tuning

FINE_TUNING_STATUS = "PENDING (not executed in Phase 1)"
# Example of what Phase 2 will run — intentionally not executed here:
# unfreeze_for_fine_tuning(student_base_model, unfreeze_from_layer=training_config["mobilenetv2"]["fine_tune_unfreeze_from_layer"])
# compile_student(student_model, learning_rate=training_config["mobilenetv2"]["fine_tune_learning_rate"])
# student_model.fit(train_ds, validation_data=val_ds, epochs=training_config["mobilenetv2"]["fine_tune_epochs"], callbacks=callbacks)
print(FINE_TUNING_STATUS)


## 14. Knowledge-Distillation readiness

- `student_model` (MobileNetV2), `baseline_model` (lightweight CNN), and `teacher_model` (ResNet50)
  all expose the **same two-logit output** (`vehicle_logit`, `pedestrian_logit`) directly — no
  Softmax, no separate pre-activation model needed, as required for a future per-target
  distillation loss (`docs/teacher_student_contract.md`).
- No real Teacher training, Teacher logits, or KD loss exists yet — `configs/training_config.json`
  keeps `distillation.enabled = false`.
- See `src/neurodriver_cnn/distillation/README.md` for the planned distillation-loss / `Distiller`
  components (computed per binary target, not over a 4-way softmax).
- **No fake KD is performed here.**


## 15. Status summary

| Component | Status |
|---|---|
| Environment/seed setup | DONE |
| Manifest loading + validation | DONE if `experiment_manifest.csv` exists, else PENDING |
| tf.data pipeline (multi-label targets) + augmentation | DONE if data available, else PENDING |
| Baseline CNN (Student 2) is lightweight (<1M params) | DONE — asserted in Section 7 (~495K total params) |
| Baseline CNN (Student 2) training | see `BASELINE_STATUS` above |
| Baseline evaluation (per-target + derived 4-state) | PENDING unless baseline trained and TEST data available |
| MobileNetV2 Student (Student 1) construction | DONE |
| MobileNetV2 forward-pass smoke test | see `MOBILENET_SMOKE_TEST_STATUS` above |
| ResNet50 Teacher construction + smoke test | see `RESNET50_STATUS` above |
| Full MobileNetV2 / ResNet50 training | NOT required for Phase 1 |
| Fine-tuning | PENDING (Phase 2+) |
| Knowledge Distillation | PENDING (Phase 2+, requires a trained ResNet50 Teacher) |
